# Limpieza de datos (Silver-Layer)

In [13]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
from pyspark.sql.functions import col, to_timestamp
path = "Files/yellow_tripdata_2015-01.csv"
df = (
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(path)
)
display(df)

StatementMeta(, c0e883ef-71f0-4d90-9a77-2954875649b7, 15, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 9e2a4c88-67d6-446f-8bdc-cbe58f2cd4ef)

In [14]:
# Silver - Limpieza de datos
# 1. Eliminar duplicados
# df_silver = df.distinct() # elimina filas duplicadas
# Como alternativa tenemos
df_silver = df.dropDuplicates() # podemos incluso eliminar duplicados por cada una de las columnas arg: subset
# 2. Eliminar vacios, en este caso se analiza que columnas deben poseer un dato, si no lo tiene, descartarlo.
columns_possible_na = ['tpep_pickup_datetime', 'tpep_dropoff_datetime', 'payment_type']
df_silver = df_silver.dropna(subset=columns_possible_na)
# 3. Tipado de Fechas (Requisito estricto para el modelo dimensional en la capa Gold)
df_silver = (
    df_silver
    .withColumn("tpep_pickup_datetime", to_timestamp(col("tpep_pickup_datetime")))
    .withColumn("tpep_dropoff_datetime", to_timestamp(col("tpep_dropoff_datetime")))
)

# 4. Eliminación de atípicos consolidada
df_silver = df_silver.filter(
    (col('pickup_longitude') != 0.0) &
    (col('pickup_latitude') != 0.0) &
    (col('dropoff_longitude') != 0.0) &
    (col('dropoff_latitude') != 0.0) &
    (col('fare_amount') > 0.0) &       # Elimina tarifas negativas o en cero
    (col('passenger_count') > 0)       # Elimina viajes sin pasajeros
)

StatementMeta(, c0e883ef-71f0-4d90-9a77-2954875649b7, 16, Finished, Available, Finished, False)

In [15]:
# Almacenar datos en tabla delta

table_name = 'tripdata_table'
(
    df_silver.write
    .format('delta')
    .mode('overwrite')
    .saveAsTable(table_name)
)

print('Guardado exitoso')

StatementMeta(, c0e883ef-71f0-4d90-9a77-2954875649b7, 17, Finished, Available, Finished, False)

Guardado exitoso


# Modelado de datos (Gold-Layer)

In [24]:
from pyspark.sql.functions import col, year, month, dayofmonth, date_format

# 1. Crear Dimensión 
# Tiempo (Extraída de la fecha de recogida)
df_dim_date = (
    df_silver.select(
        col("tpep_pickup_datetime").alias("date_id"),
        year("tpep_pickup_datetime").alias("year"),
        month("tpep_pickup_datetime").alias("month"),
        dayofmonth("tpep_pickup_datetime").alias("day"),
        date_format("tpep_pickup_datetime", "EEEE").alias("day_name")
    )
    .dropDuplicates(["date_id"]) # Aseguramos que cada fecha sea única
)

# Guardar la Dimensión como Tabla Delta Gold
(
    df_dim_date.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("gold_dim_date")
)
# Vendedor
df_dim_vendor = spark.createDataFrame(
    data = [(1,'Creative Mobile Technologies'), (2 , 'VeriFone Inc.')],
    schema = ['vendor_id', 'vendor']
)

(
    df_dim_vendor.write
    .format('delta')
    .mode('overwrite')
    .saveAsTable('gold_dim_vendor')
)

# RateCodeID

df_dim_rate_code = spark.createDataFrame(
    data = [(1, 'Standar rate'), (2, 'JFK'), (3, 'Newark'),(4, 'Nassau or Westchester'), (5, 'Negotiated fare'), (6, 'Group ride')],
    schema = ['rate_code_id', 'rate']
)

(
    df_dim_rate_code.write
    .format('delta')
    .mode('overwrite')
    .saveAsTable('gold_dim_rate')
)

# PaymentType

df_dim_payment = spark.createDataFrame(
    data = [(1, 'Credit Card'), (2, 'Cash'), (3, 'No charge'), (4, 'Dispute'), (5, 'Unknown'), (6, 'Voided trip')],
    schema = ['payment_id', 'payment']
)
(
    df_dim_payment.write
    .format('delta')
    .mode('overwrite')
    .saveAsTable('gold_dim_payment')
)

print('Dimensiones creadas con exito')

StatementMeta(, c0e883ef-71f0-4d90-9a77-2954875649b7, 27, Finished, Available, Finished, False)

Dimensiones creadas con exito


In [22]:
# 2. Crear Tabla de Hechos (Viajes)
# Seleccionamos las llaves foráneas y las métricas, descartando texto innecesario
df_fact_trips = df_silver.select(
    col("tpep_pickup_datetime").alias("date_id"), # Llave foránea hacia dim_date
    col("RateCodeID").alias("rade_code_id"),
    col("VendorID").alias("vendor_id"),
    col("payment_type").alias("payment_type_id"),
    col("passenger_count"),
    col("trip_distance"),
    col("fare_amount"),
    col("tpep_dropoff_datetime"),
    col("pickup_longitude"),
    col("pickup_latitude"),
    col("dropoff_longitude"),
    col("dropoff_latitude"),
    col("tip_amount"),
    col("tolls_amount"),
    col("total_amount")
)
display(df_fact_trips)
# Guardar la Tabla de Hechos como Tabla Delta Gold
(
    df_fact_trips.write
    .format("delta")
    .mode("overwrite")
    #.option("mergeSchema", "true") # Una vez creado el esquema nos permite cambiarlo
    .saveAsTable("gold_fact_trips") # Siempre al final
)

print("¡Modelo estrella físico (Capa Gold) creado con éxito!")

StatementMeta(, c0e883ef-71f0-4d90-9a77-2954875649b7, 24, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c012484a-b36d-4555-b934-1655403f42d0)

¡Modelo estrella físico (Capa Gold) creado con éxito!
